#### Scope
In this notebook code used to produce figures and tables is made available.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Expected columns (as in your dataset)
MODEL_COLS = {
    "Roots Screening": {
        "precision": "precision_roots_screening",
        "recall": "recall_roots_screening",
        "mse": "mse_roots_screening",
        "shd": "shd_roots_screening",
        "exec_time_sec": "exec_time_sec_roots_screening",
    },
    "DirectLiNGAM": {
        "precision": "precision_direct_lingam",
        "recall": "recall_direct_lingam",
        "mse": "mse_direct_lingam",
        "shd": "shd_direct_lingam",
        "exec_time_sec": "exec_time_sec_direct_lingam",
    },
}

VARS = ["n", "p", "m"]

def to_long(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Convert wide df into long format for a single metric with columns:
    [n, p, m, random_state, model, value]
    """
    rows = []
    for model_name, mapping in MODEL_COLS.items():
        col = mapping[metric]
        if col not in df.columns:
            raise KeyError(f"Missing column for metric '{metric}': {col}")
        tmp = df[["n", "p", "m", "random_state"]].copy()
        tmp["model"] = model_name
        tmp["value"] = df[col]
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    return long_df

def _filter_exact(df_long: pd.DataFrame, filters: dict | None) -> pd.DataFrame:
    d = df_long.copy()
    if filters:
        for k, v in filters.items():
            if isinstance(v, (list, tuple, set)):
                d = d[d[k].isin(list(v))].copy()
            else:
                d = d[d[k] == v].copy()
    return d


def plot_runtime_main_figure_plotly(
    df: pd.DataFrame,
    n_fixed: int | None = None,             # e.g., 2000; if None, uses mode
    p_values: list[int] | None = None,      # optional: restrict to [100,300,500]
    out_pdf="fig_root_runtime_main.pdf",
    out_png="fig_root_runtime_main.png",
    debug_points: bool = False,
):
    """
    MAIN PAPER FIGURE:
      - runtime vs p
      - n fixed (default = mode)
      - pooled over m (no facets)
      - log-scale y
    """
    # choose n
    if n_fixed is None:
        n_fixed = int(df["n"].mode().iloc[0])

    dlong = to_long(df, "exec_time_sec")  # columns: n,p,m,random_state,model,value
    d = _filter_exact(dlong, {"n": n_fixed})

    if p_values is not None:
        d = d[d["p"].isin(p_values)].copy()

    # guard against zeros (log scale). If you truly can have 0, add tiny epsilon.
    d = d[d["value"].notna()].copy()
    if (d["value"] <= 0).any():
        eps = 1e-6
        d.loc[d["value"] <= 0, "value"] = eps

    hover_cols = ["model", "n", "p", "m", "random_state", "value"]
    fig = px.box(
        d,
        x="p",
        y="value",
        color="model",
        points=("all" if debug_points else False),
        labels={"p": "Number of variables (p)", "value": "Execution time (s, log10 scale)", "model": ""},
        title="",
        color_discrete_map={
            "DirectLiNGAM": "#808080",
            "Roots Screening": "#009fd4",
        },
        hover_data=hover_cols if debug_points else None,
    )

    # group boxes side-by-side
    fig.update_layout(
        template="simple_white",
        boxmode="group",
        title_x=0.5,
        legend_title_text="",
        font=dict(size=14),
        margin=dict(l=70, r=30, t=40, b=60),
        legend=dict(
        x=0.02,
        y=0.98,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.0)",
        borderwidth=0,
        font=dict(size=13),
        itemsizing="constant",
    ),

    )

    # log y-axis
    fig.update_yaxes(
        type="log",
        tickmode="array",
        tickvals=[
            100, 200, 500,
            1_000, 2_000, 5_000,
            10_000, 20_000, 50_000,
            100_000
        ],
        ticktext=[
            "100", "200", "500",
            "1k", "2k", "5k",
            "10k", "20k", "50k",
            "100k"
        ],
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
    )
    
    # Optional: a single global x label is already fine here (no facets), so no annotation needed.

    # export
    fig.write_image(out_pdf)
    fig.write_image(out_png, scale=3)

    return fig

def plot_shd_main_figure_plotly(
    df: pd.DataFrame,
    n_fixed: int = 6000,
    p_values: list[int] | None = None,
    out_pdf="fig_root_shd_main.pdf",
    out_png="fig_root_shd_main.png",
    debug_points: bool = False,
):
    """
    MAIN PAPER FIGURE (Panel B):
      - SHD vs p
      - n fixed
      - pooled over m
      - linear y-scale
    """
    dlong = to_long(df, "shd")  # columns: n,p,m,random_state,model,value
    d = dlong[dlong["n"] == n_fixed].copy()

    if p_values is not None:
        d = d[d["p"].isin(p_values)].copy()

    hover_cols = ["model", "n", "p", "m", "random_state", "value"]

    fig = px.box(
        d,
        x="p",
        y="value",
        color="model",
        points=("all" if debug_points else False),
        labels={
            "p": "Number of variables (p)",
            "value": "Structural Hamming Distance (SHD)",
            "model": "",
        },
        title="",
        color_discrete_map={
            "DirectLiNGAM": "#808080",
            "Roots Screening": "#009fd4",
        },
        hover_data=hover_cols if debug_points else None,
    )

    fig.update_layout(
        template="simple_white",
        boxmode="group",
        title_x=0.5,
        # update layout showlegend = True in case is needed
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.0)",
            borderwidth=0,
            font=dict(size=13),
            itemsizing="constant",
        ),
        font=dict(size=14),
        margin=dict(l=70, r=30, t=40, b=60),
    )

    
    fig.update_yaxes(
    tickmode="array",
    tickvals= [0,10,20,30,40,50,75,100,125,150,175,200],#[0, 250, 500, 750, 1000, 1250],
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    )
    fig.update_xaxes(showgrid=False)
    fig.update_layout(showlegend=False)

    fig.write_image(out_pdf)
    fig.write_image(out_png, scale=3)

    return fig


In [2]:
df = pd.read_csv("./execution_time_comparison/exp3/root_screening_lingam_ba_network_exp3.csv")

In [18]:
plot_runtime_main_figure_plotly(df = df, n_fixed=2000, p_values=[100,300,500], debug_points=False)

In [3]:
df[(df["p"]==500) & (df["n"]==6000) & (df["m"]==1)]

,precision_roots_screening,recall_roots_screening,mse_roots_screening,shd_roots_screening,precision_direct_lingam,recall_direct_lingam,mse_direct_lingam,shd_direct_lingam,p,m,n,random_state,exec_time_sec_roots_screening,exec_time_sec_direct_lingam
380,0.972710,1.0,4.050547e-07,14,0.965184,1.0,4.402773e-07,18,500,1,6000,0,45900.079534,82708.999356
381,0.970817,1.0,3.946878e-07,15,0.952290,1.0,5.446808e-07,25,500,1,6000,1,49952.566367,84454.410859
382,0.972710,1.0,4.351326e-07,14,0.945076,1.0,5.673955e-07,29,500,1,6000,2,44803.999455,82862.133281
383,0.990079,1.0,3.175461e-07,5,0.963320,1.0,3.927848e-07,19,500,1,6000,3,46688.708231,82839.146969
384,0.998000,1.0,2.960160e-07,1,0.961464,1.0,4.697344e-07,20,500,1,6000,4,47677.028686,86779.760367
385,0.994024,1.0,3.326747e-07,3,0.976517,1.0,4.271494e-07,12,500,1,6000,5,48890.999118,86524.860001
386,0.972710,1.0,3.948616e-07,14,0.948669,1.0,5.137871e-07,27,500,1,6000,6,48916.656852,87308.929655
387,0.984221,1.0,3.970817e-07,8,0.970817,1.0,4.585928e-07,15,500,1,6000,7,46781.318480,84218.436755
388,0.990079,1.0,3.151750e-07,5,0.974609,1.0,3.328025e-07,13,500,1,6000,8,47448.668845,83584.402910
389,0.990079,1.0,3.298018e-07,5,0.955939,1.0,4.594694e-07,23,500,1,6000,9,45800.705021,81822.642380


In [23]:
plot_shd_main_figure_plotly(df = df, n_fixed = 2_000, p_values= [100,300,500], debug_points=False)

In [9]:

def make_roots_screening_summary_table(
    df: pd.DataFrame,
    n_fixed: int = 6000,
    p_values=(100, 300, 500),
    pool_over_m: bool = True,   # kept for clarity; pooling is implicit by default
    formatted: bool = True,
):
    """
    Summary table (mean ± std) comparing Roots Screening vs DirectLiNGAM
    at fixed n, varying p. Results are pooled across m (and seeds) by default.

    Returns:
      - if formatted=True: a table with pretty strings "mean ± std"
      - else: a numeric table with mean/std columns (useful for custom LaTeX)
    """
    df = df.copy()

    # ---- sanity checks ----
    required_base = {"n", "p"}
    missing_base = required_base - set(df.columns)
    if missing_base:
        raise KeyError(f"Missing required columns: {sorted(missing_base)}")

    # Check metric columns exist
    required_metric_cols = []
    for _, mapping in MODEL_COLS.items():
        required_metric_cols.extend(mapping.values())
    required_metric_cols = set(required_metric_cols)
    missing_metric = required_metric_cols - set(df.columns)
    if missing_metric:
        raise KeyError(
            "Missing metric columns in dataframe:\n"
            + "\n".join(sorted(missing_metric))
            + "\n\nTip: print([c for c in df.columns if 'exec_time' in c]) and update MODEL_COLS."
        )

    # ---- filter to the regime you want in the paper ----
    d = df[df["n"] == n_fixed].copy()
    d = d[d["p"].isin(p_values)].copy()
    if d.empty:
        raise ValueError(
            f"No rows after filtering to n={n_fixed} and p in {list(p_values)}. "
            "Check your df['n'] and df['p'] values."
        )

    rows = []
    for method, mapping in MODEL_COLS.items():
        for p in sorted(p_values):
            sub = d[d["p"] == p].copy()

            out = {"Method": method, "p": int(p), "N": int(len(sub))}

            # aggregate each metric
            for metric_name, col in mapping.items():
                vals = sub[col].dropna()
                out[f"{metric_name}_mean"] = float(vals.mean()) if len(vals) else np.nan
                out[f"{metric_name}_std"] = float(vals.std(ddof=1)) if len(vals) else np.nan

            # derived F1 from precision/recall means (consistent with summary tables)
            pr = out["precision_mean"]
            rc = out["recall_mean"]
            out["f1_mean"] = (2 * pr * rc / (pr + rc)) if (np.isfinite(pr) and np.isfinite(rc) and (pr + rc) > 0) else np.nan

            rows.append(out)

    numeric = pd.DataFrame(rows).sort_values(["p", "Method"]).reset_index(drop=True)
    

    if not formatted:
        return numeric

    # ---- pretty formatting for LaTeX ----
    def fmt(m, s, digits=3):
        if pd.isna(m):
            return ""
        if pd.isna(s):
            return f"{m:.{digits}f}"
        return f"{m:.{digits}f} ± {s:.{digits}f}"

    pretty = pd.DataFrame({
        "p": numeric["p"],
        "Method": numeric["Method"],
        "SHD": [fmt(m, s, digits=1) for m, s in zip(numeric["shd_mean"], numeric["shd_std"])],
        "MSE": [fmt(m, s, digits=7) for m, s in zip(numeric["mse_mean"], numeric["mse_std"])],
        "Precision": [fmt(m, s, digits=3) for m, s in zip(numeric["precision_mean"], numeric["precision_std"])],
        "Recall": [fmt(m, s, digits=3) for m, s in zip(numeric["recall_mean"], numeric["recall_std"])],
        # F1 in many papers is shown without ± (it’s derived); keep it compact
        "F1": [("" if pd.isna(m) else f"{m:.3f}") for m in numeric["f1_mean"]],
        # exec time: usually large → fewer decimals
        "Exec. time (s)": [fmt(m, s, digits=0) for m, s in zip(numeric["exec_time_sec_mean"], numeric["exec_time_sec_std"])],
        "N": numeric["N"],
    }).sort_values(["p", "Method"]).reset_index(drop=True)

    return pretty

In [27]:
table_roots_screening = make_roots_screening_summary_table(df,n_fixed=2000, p_values=[100,300,500])

In [28]:
table_roots_screening

,p,Method,SHD,MSE,Precision,Recall,F1,Exec. time (s),N
0,100,DirectLiNGAM,5.5 ± 4.6,0.0000426 ± 0.0002544,0.969 ± 0.022,1.000 ± 0.001,0.984,365 ± 19,60
1,100,Roots Screening,2.9 ± 2.3,0.0000087 ± 0.0000028,0.983 ± 0.015,1.000 ± 0.000,0.992,192 ± 26,60
2,300,DirectLiNGAM,28.7 ± 16.3,0.0000258 ± 0.0000836,0.949 ± 0.025,1.000 ± 0.001,0.974,9831 ± 471,60
3,300,Roots Screening,31.1 ± 72.8,0.0004910 ± 0.0034168,0.959 ± 0.058,0.999 ± 0.003,0.979,4829 ± 644,60
4,500,DirectLiNGAM,91.0 ± 123.8,0.0001335 ± 0.0004629,0.926 ± 0.051,0.999 ± 0.006,0.961,45783 ± 1898,60
5,500,Roots Screening,81.3 ± 190.4,0.0001629 ± 0.0007301,0.947 ± 0.072,0.999 ± 0.006,0.972,22099 ± 3006,60


In [29]:
table_roots_screening.drop(columns=["N"])

,p,Method,SHD,MSE,Precision,Recall,F1,Exec. time (s)
0,100,DirectLiNGAM,5.5 ± 4.6,0.0000426 ± 0.0002544,0.969 ± 0.022,1.000 ± 0.001,0.984,365 ± 19
1,100,Roots Screening,2.9 ± 2.3,0.0000087 ± 0.0000028,0.983 ± 0.015,1.000 ± 0.000,0.992,192 ± 26
2,300,DirectLiNGAM,28.7 ± 16.3,0.0000258 ± 0.0000836,0.949 ± 0.025,1.000 ± 0.001,0.974,9831 ± 471
3,300,Roots Screening,31.1 ± 72.8,0.0004910 ± 0.0034168,0.959 ± 0.058,0.999 ± 0.003,0.979,4829 ± 644
4,500,DirectLiNGAM,91.0 ± 123.8,0.0001335 ± 0.0004629,0.926 ± 0.051,0.999 ± 0.006,0.961,45783 ± 1898
5,500,Roots Screening,81.3 ± 190.4,0.0001629 ± 0.0007301,0.947 ± 0.072,0.999 ± 0.006,0.972,22099 ± 3006
